In [13]:
import pandas as pd
df=pd.read_csv(r"df_final_demo.txt")
part1=pd.read_csv(r"df_final_web_data_pt_1.txt")
part2=pd.read_csv(r"df_final_web_data_pt_2.txt")
df_concated=pd.concat([part1,part2], ignore_index=True)
Variation=pd.read_csv(r"df_final_experiment_clients.txt")
merged = pd.merge(df, df_concated, on='client_id', how='inner')
full_df = pd.merge(merged, Variation, on='client_id', how='inner')

In [14]:
full_df["clnt_age"]=full_df["clnt_age"].round(0)


In [15]:
full_df['gendr'] = full_df['gendr'].replace('X', 'U').fillna('U')


In [16]:
test_table=full_df[full_df['Variation'].str.lower() == 'test']


In [17]:
#droping duplicates
test_table=test_table.drop_duplicates()

In [18]:
#dropping the null value
test_table =test_table.dropna()

In [19]:
control_table = full_df[full_df['Variation'].str.lower() == 'control']

In [20]:
#droping duplicates
control_table=control_table.drop_duplicates()

In [21]:
#dropping the null value
control_table =control_table.dropna()

In [22]:
#completion status YES/NO---TEST

#completion status YES/NO
completed_clients = test_table[test_table["process_step"] == "confirm"]["client_id"].unique()

test_table['completed_flag'] = test_table['client_id'].isin(completed_clients).astype(int)
test_table['completed_flag'] = test_table['completed_flag'].map({1: 'yes', 0: 'no'})

In [23]:
#completion status YES/NO---CONTROL
#completion status YES/NO
completed_clients = control_table[control_table["process_step"] == "confirm"]["client_id"].unique()
control_table['completed_flag'] = control_table['client_id'].isin(completed_clients).astype(int)
control_table['completed_flag'] =control_table['completed_flag'] .map({1: 'yes', 0: 'no'})


In [24]:
#time spent on each step
test_table = test_table.copy()

# Step 2: Convert date_time in test_table, not merged
test_table['date_time'] = pd.to_datetime(test_table['date_time'], errors='coerce')

# Step 3: Sort and calculate time spent
sorted_test = test_table.sort_values(by=['visit_id', 'date_time'])

test_table['time_spent_minutes'] = (
    sorted_test.groupby('visit_id')['date_time']
    .diff()
    .abs()
    .dt.total_seconds() / 60
    
)



In [25]:
mean = test_table['time_spent_minutes'].mean()
test_table['time_spent_minutes'].fillna(mean, inplace=True)
test_table['time_spent_minutes']=test_table['time_spent_minutes'].round(2)

C:\Users\prodd\AppData\Local\Temp\ipykernel_25940\2205890008.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_table['time_spent_minutes'].fillna(mean, inplace=True)


In [26]:
Q1 = test_table['time_spent_minutes'].quantile(0.25)
Q3 = test_table['time_spent_minutes'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

test_table = test_table[(test_table['time_spent_minutes'] >= lower_bound) & (test_table['time_spent_minutes'] <= upper_bound)]

In [27]:
# Make a clean copy
control_table = control_table.copy()

# Step 2: Convert date_time in test_table, not merged
control_table['date_time'] = pd.to_datetime(control_table['date_time'], errors='coerce')

# Step 3: Sort and calculate time spent
sorted_control = control_table.sort_values(by=['visit_id', 'date_time','client_id'])

control_table['time_spent_minutes'] = (
    sorted_control.groupby('visit_id')['date_time']
    .diff()
    .abs()
    .dt.total_seconds() / 60
)

In [28]:
mean = control_table['time_spent_minutes'].mean()
control_table['time_spent_minutes'].fillna(mean, inplace=True)
control_table['time_spent_minutes']=control_table['time_spent_minutes'].round(2)

C:\Users\prodd\AppData\Local\Temp\ipykernel_25940\4207991998.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  control_table['time_spent_minutes'].fillna(mean, inplace=True)


In [29]:
Q1 = control_table['time_spent_minutes'].quantile(0.25)
Q3 = control_table['time_spent_minutes'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
control_table= control_table[(control_table['time_spent_minutes'] >= lower_bound) & (control_table['time_spent_minutes'] <= upper_bound)]

In [30]:
step_order = {
    'start': 1,
    'step_1': 2,
    'step_2': 3,
    'step_3': 4,
    'confirm': 5
}


In [31]:
def individual_error_rate(df, step_order):
    df = df.replace({'process_step': step_order})
    df = df.sort_values(['client_id', 'visit_id', 'date_time'])
    df['prev_step'] = df.groupby(['client_id', 'visit_id'])['process_step'].shift()
    df['backward_move'] = df['process_step'] < df['prev_step']
    return df.groupby('client_id')['backward_move'].mean().reset_index(name='error_rate')

test_individual = individual_error_rate(test_table, step_order)
control_individual = individual_error_rate(control_table, step_order)

C:\Users\prodd\AppData\Local\Temp\ipykernel_25940\3720362767.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({'process_step': step_order})
C:\Users\prodd\AppData\Local\Temp\ipykernel_25940\3720362767.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({'process_step': step_order})


In [32]:
test_individual['error_rate']=test_individual['error_rate'].apply(lambda x: f"{round(x * 100, 2)}%")

In [33]:
control_individual['error_rate']=control_individual['error_rate'].apply(lambda x: f"{round(x * 100, 2)}%")

In [34]:
test_table=pd.merge(test_table, test_individual, on='client_id', how='inner')

In [35]:
control_table=pd.merge(control_table, control_individual, on='client_id', how='inner')

In [36]:
final_clean=pd.concat([test_table,control_table],ignore_index=True)

In [37]:
final_clean.columns = final_clean.columns.str.replace(' ', '_').str.lower()

In [38]:
final_clean

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,visitor_id,visit_id,process_step,date_time,variation,completed_flag,time_spent_minutes,error_rate
0,836976,6.0,73.0,60.0,U,2.0,45105.30,6.0,9.0,427070339_1413275162,228976764_46825473280_96584,confirm,2017-04-02 11:47:50,Test,yes,1.08,0.0%
1,836976,6.0,73.0,60.0,U,2.0,45105.30,6.0,9.0,427070339_1413275162,228976764_46825473280_96584,step_3,2017-04-02 11:23:08,Test,yes,0.73,0.0%
2,836976,6.0,73.0,60.0,U,2.0,45105.30,6.0,9.0,427070339_1413275162,228976764_46825473280_96584,step_2,2017-04-02 11:22:24,Test,yes,0.77,0.0%
3,836976,6.0,73.0,60.0,U,2.0,45105.30,6.0,9.0,427070339_1413275162,228976764_46825473280_96584,step_1,2017-04-02 11:21:38,Test,yes,0.17,0.0%
4,836976,6.0,73.0,60.0,U,2.0,45105.30,6.0,9.0,427070339_1413275162,228976764_46825473280_96584,start,2017-04-02 11:21:28,Test,yes,1.41,0.0%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293214,6967120,21.0,260.0,68.0,M,3.0,4279873.38,6.0,9.0,663474827_38847225720,923772865_74694577233_449836,confirm,2017-04-12 19:06:55,Control,yes,1.68,0.0%
293215,6967120,21.0,260.0,68.0,M,3.0,4279873.38,6.0,9.0,663474827_38847225720,923772865_74694577233_449836,step_3,2017-04-12 19:05:14,Control,yes,0.75,0.0%
293216,6967120,21.0,260.0,68.0,M,3.0,4279873.38,6.0,9.0,663474827_38847225720,923772865_74694577233_449836,step_2,2017-04-12 19:04:29,Control,yes,0.37,0.0%
293217,6967120,21.0,260.0,68.0,M,3.0,4279873.38,6.0,9.0,663474827_38847225720,923772865_74694577233_449836,step_1,2017-04-12 19:04:07,Control,yes,0.52,0.0%


In [39]:
final_clean.to_csv("final_clean.csv", index=False)